# LSTM Model Training for WESAD Dataset

This notebook trains an LSTM model to classify stress, amusement, and baseline states using the WESAD dataset.

In [1]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import glob

## 1. Configuration and Setup

In [8]:
# Configuration
DATASET_PATH = 'Dataset/WESAD'
TARGET_USERS = ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']
TARGET_LABELS = {1: 0, 2: 1, 3: 2} # 1: baseline, 2: stress, 3: amusement -> 0, 1, 2
N_FEATURES = 6 # ACC (3), BVP (1), EDA (1), TEMP (1)
DOWNSAMPLE_RATE = 4 # Hz, the rate to downsample all signals to
WINDOW_SIZE_SEC = 60 # seconds
STRIDE_SEC = 1 # seconds

# LSTM Hyperparameters
HIDDEN_DIM = 128
LAYER_DIM = 2
OUTPUT_DIM = len(TARGET_LABELS)
BATCH_SIZE = 64
NUM_EPOCHS = 20
LEARNING_RATE = 0.001

# Setup device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 2. Data Loading and Preprocessing

In [ ]:
def load_and_preprocess_data(subject_path):
    """Loads a single subject's data, downsamples, and synchronizes."""
    with open(subject_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    # --- Extract Wrist Data --- #
    wrist_data = data['signal']['wrist']
    acc = wrist_data['ACC']
    bvp = wrist_data['BVP']
    eda = wrist_data['EDA']
    temp = wrist_data['TEMP']
    labels = data['label']

    # --- Downsample --- #
    # Original sampling rates: ACC=32Hz, BVP=64Hz, EDA=4Hz, TEMP=4Hz, label=700Hz
    acc_down = acc[::32 // DOWNSAMPLE_RATE]
    bvp_down = bvp[::64 // DOWNSAMPLE_RATE]
    eda_down = eda[::4 // DOWNSAMPLE_RATE]
    temp_down = temp[::4 // DOWNSAMPLE_RATE]
    
    # Align labels by resampling at the data points' timestamps
    label_timestamps = np.arange(0, len(labels) / 700.0, 1/700.0)
    data_timestamps = np.arange(0, len(acc_down) / DOWNSAMPLE_RATE, 1/DOWNSAMPLE_RATE)
    idx = np.searchsorted(label_timestamps, data_timestamps, side='left')
    idx = np.clip(idx, 0, len(labels) - 1)
    labels_down = labels[idx].astype(int)
    
    # Find the minimum length to truncate all signals
    min_len = min(len(acc_down), len(bvp_down), len(eda_down), len(temp_down), len(labels_down))
    
    # --- Combine Features --- #
    # Shape: (min_len, N_FEATURES)
    features = np.concatenate([
        acc_down[:min_len],
        bvp_down[:min_len],
        eda_down[:min_len],
        temp_down[:min_len]
    ], axis=1)
    
    labels_final = labels_down[:min_len]
    
    return features, labels_final

def create_windows(features, labels):
    """Creates sliding windows of data and corresponding labels."""
    window_samples = WINDOW_SIZE_SEC * DOWNSAMPLE_RATE
    stride_samples = STRIDE_SEC * DOWNSAMPLE_RATE

    X, y = [], []
    for i in range(0, len(features) - window_samples, stride_samples):
        window_features = features[i : i + window_samples]
        window_labels = labels[i : i + window_samples]
        
        # Use the most frequent label in the window as the segment's label
        most_frequent_label = np.bincount(window_labels).argmax()
        
        if most_frequent_label in TARGET_LABELS:
            X.append(window_features)
            y.append(TARGET_LABELS[most_frequent_label])

    return np.array(X), np.array(y)


In [10]:
# --- Process all subjects --- #
all_X, all_y = [], []
for user in TARGET_USERS:
    subject_path = os.path.join(DATASET_PATH, user, f'{user}.pkl')
    if os.path.exists(subject_path):
        print(f'Processing {user}...')
        features, labels = load_and_preprocess_data(subject_path)
        X, y = create_windows(features, labels)
        all_X.append(X)
        all_y.append(y)

X_combined = np.concatenate(all_X, axis=0)
y_combined = np.concatenate(all_y, axis=0)

print(f'\nTotal windows created: {len(X_combined)}')
print(f'Feature shape: {X_combined.shape}')
print(f'Label distribution: {np.bincount(y_combined)}')

# --- Normalize Features --- #
# Reshape for scaler: (num_samples * window_length, num_features)
scaler = StandardScaler()
X_reshaped = X_combined.reshape(-1, N_FEATURES)
X_scaled_reshaped = scaler.fit_transform(X_reshaped)
X_scaled = X_scaled_reshaped.reshape(X_combined.shape)

# --- Split Data --- #
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_combined, test_size=0.2, random_state=42, stratify=y_combined
)
print(f'\nTrain set size: {len(X_train)}')
print(f'Test set size: {len(X_test)}')


Processing S2...


TypeError: interp() got an unexpected keyword argument 'method'

## 3. PyTorch Dataset and DataLoader

In [ ]:
class WesadDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = WesadDataset(X_train, y_train)
test_dataset = WesadDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 4. LSTM Model Definition

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        
        # Use nn.LSTM which is more standard and efficient
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True, dropout=0.3)
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # Initialize hidden and cell states
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(device).requires_grad_()
        c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(device).requires_grad_()
        
        # We need to detach as we are doing truncated backpropagation through time (BPTT)
        # If we don't, we'll backprop all the way to the start of the sequence
        out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))
        
        # Index hidden state of last time step
        out = self.fc(out[:, -1, :]) 
        return out


## 5. Model Training

In [ ]:
model = LSTMModel(N_FEATURES, HIDDEN_DIM, LAYER_DIM, OUTPUT_DIM)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Starting training...")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for i, (sequences, labels) in enumerate(train_loader):
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        
        # Backward and optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * sequences.size(0)
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_acc = (correct_predictions / total_samples) * 100
    
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%')

## 6. Model Evaluation

In [ ]:
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        outputs = model(sequences)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = 100 * correct / total
print(f'\nTest Accuracy: {accuracy:.2f} %')

# Optional: Print classification report for more details
try:
    from sklearn.metrics import classification_report
    report = classification_report(all_labels, all_preds, target_names=['baseline', 'stress', 'amusement'])
    print("\nClassification Report:")
    print(report)
except ImportError:
    print("\nPlease install scikit-learn to see the classification report: pip install -U scikit-learn")